# KMeans 개선안 5: HOG 특징 사용

원본 픽셀값 대신 이미지의 윤곽선과 방향 정보를 요약하는 HOG(Histogram of Oriented Gradients) 특징을 추출해 KMeans에 사용합니다.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

data_path = Path('fruits_300.npy')
if not data_path.exists():
    urlretrieve('https://bit.ly/fruits_300_data', data_path)


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog

fruits = np.load('fruits_300.npy') / 255.0


In [3]:
def extract_hog_features(images):
    return np.array([
        hog(
            image,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm='L2-Hys',
            feature_vector=True,
        )
        for image in images
    ])

hog_features = extract_hog_features(fruits)
hog_features_scaled = StandardScaler().fit_transform(hog_features)

print('HOG feature shape:', hog_features_scaled.shape)


HOG feature shape: (300, 4356)


In [4]:
km = KMeans(n_clusters=3, n_init=20, random_state=42)
labels = km.fit_predict(hog_features_scaled)

print('labels:', np.unique(labels, return_counts=True))
print('silhouette:', silhouette_score(hog_features_scaled, labels))
print('davies_bouldin:', davies_bouldin_score(hog_features_scaled, labels))
print('calinski_harabasz:', calinski_harabasz_score(hog_features_scaled, labels))


labels: (array([0, 1, 2], dtype=int32), array([100, 100, 100]))
silhouette: 0.16661541097261076
davies_bouldin: 2.0622977940411835
calinski_harabasz: 51.00375845438823


In [5]:
scores = []

for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(hog_features_scaled)
    scores.append({
        'k': k,
        'inertia': km.inertia_,
        'silhouette': silhouette_score(hog_features_scaled, labels),
        'davies_bouldin': davies_bouldin_score(hog_features_scaled, labels),
        'calinski_harabasz': calinski_harabasz_score(hog_features_scaled, labels),
    })

for row in scores:
    print(
        f"k={row['k']}, inertia={row['inertia']:.2f}, "
        f"silhouette={row['silhouette']:.3f}, "
        f"DB={row['davies_bouldin']:.3f}, "
        f"CH={row['calinski_harabasz']:.2f}"
    )


k=2, inertia=1138323.44, silhouette=0.123, DB=2.314, CH=44.11
k=3, inertia=972712.50, silhouette=0.167, DB=2.062, CH=51.00
k=4, inertia=895252.42, silhouette=0.194, DB=2.135, CH=45.36
k=5, inertia=839853.78, silhouette=0.213, DB=1.697, CH=41.00
k=6, inertia=801776.91, silhouette=0.217, DB=1.835, CH=37.04
k=7, inertia=773044.80, silhouette=0.227, DB=1.693, CH=33.72
k=8, inertia=749301.74, silhouette=0.166, DB=1.920, CH=31.04
k=9, inertia=731485.90, silhouette=0.131, DB=2.300, CH=28.61
k=10, inertia=709921.76, silhouette=0.137, DB=2.226, CH=27.09


In [7]:
import matplotlib.pyplot as plt

def draw_fruits(arr, ratio=1):
    n = len(arr)    # n은 샘플 개수입니다
    # 한 줄에 10개씩 이미지를 그립니다. 샘플 개수를 10으로 나누어 전체 행 개수를 계산합니다.
    rows = int(np.ceil(n/10))
    # 행이 1개 이면 열 개수는 샘플 개수입니다. 그렇지 않으면 10개입니다.
    cols = n if rows < 2 else 10
    fig, axs = plt.subplots(rows, cols,
                            figsize=(cols*ratio, rows*ratio), squeeze=False)
    for i in range(rows):
        for j in range(cols):
            if i*10 + j < n:    # n 개까지만 그립니다.
                axs[i, j].imshow(arr[i*10 + j], cmap='gray_r')
            axs[i, j].axis('off')
    plt.show()

## 클러스터 중심

In [8]:
draw_fruits(km.cluster_centers_.reshape(-1, 100, 100), ratio=3)

ValueError: cannot reshape array of size 43560 into shape (100,100)